# 第2回 演習: LoRA で回答形式を学習し、学習前後を比較する

架空の講座 FAQ を題材に、小さな指示チューニング済みモデルを LoRA で短時間学習し、学習の前後で **何が変わり、何が変わらないか** を比べます。評価は **形式遵守率**（`answer` と `needs_review` の JSON を守れたか）と **内容正解率** に分けます。

**この演習で確かめること**

1. プロンプトで形式を指示しても、小さなモデルは守れないことがある（セクション 3、学習前）。
2. LoRA で数分学習すると、形式はほぼ守れるようになる（セクション 5、学習後）。
3. 形式が整っても、FAQ に無いことを聞かれたときの判断（確認を求めるか、断定してしまうか）までは保証されない（セクション 5、6）。
4. 発展: 同じモデルを 4bit 量子化で読み込むと、学習しなくても出力が変わる（セクション 8）。

**進め方**

| セクション | 内容 | 所要時間の目安（T4） |
| --- | --- | --- |
| 0〜3 | 環境、設定、データ生成、モデル読み込み、学習前評価 | 約 5 分 |
| 4〜7 | 学習、学習後評価、知識質問、比較表と提出 | 約 10 分（学習 4〜5 分を含む） |
| 8（任意） | 4bit 量子化で読み込んで比べる | 約 2 分 |

授業前に 0〜3 を実行して環境が動くことを確かめておくと、当日の詰まりが減ります（任意）。Colab はセッションが切れると変数が消えるので、その場合は当日あらためて 0 から実行します。

**使うもの**: Google Colab（T4 GPU、無料枠）。モデルは `Qwen/Qwen3.5-0.8B`（Apache-2.0）。データはこのノートブックが生成する架空データで、実在の締切や URL は含みません。

**読み方**: 各セクションの説明に「やること」「見るところ」「うまくいかないとき」を書いています。セルは上から順に実行してください。途中のセルを飛ばすと、後のセルで変数が見つからないエラーになります。

版: 1.1（2026-10-18）

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.14.1+cu130'}


## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。
- `num_train_epochs: 20` は、学習データ（14 件）を 20 周する設定です。1 周が 2 ステップなので合計 40 ステップで、T4 で学習が 4〜5 分に収まります。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。

**見るところ**: 設定の JSON が表示されます。


In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    # "lr": 1e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    # "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
    "drive_dir": "/content/drive/MyDrive/aie2026/runs", # 実験ごとに保存に変更
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = True  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

Mounted at /content/drive
{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 20,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 96,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/runs"
}


## 2. 架空 FAQ と教師データの生成

**やること**: 5 項目の架空 FAQ から、学習用（train）、検証用（valid）、評価用（test）のデータを作ります。

モデルへの入力は「FAQ ＋ 質問」、望ましい出力は `{"answer": <文字列>, "needs_review": <真偽値>}` の JSON 1 行です。

- FAQ に根拠がある質問には、FAQ の文で答え、`needs_review: false` にする。
- FAQ に無い質問には、「この FAQ には記載がありません。運営に確認が必要です。」と答え、`needs_review: true` にする。

**データの分け方**

- 同じ FAQ 項目を聞く質問の言い換えを、**テンプレート単位**で train / valid / test に振り分けます。学習で見た文がそのまま test に出ないようにするためです。
- test は 21 問（根拠あり 15、根拠なし 6）です。件数を確保するため、test にだけ使う質問を別に用意しています。
- 正解（`needs_review` の値と、答えに含まれるべき語句）は `gold` に分けて持ち、モデルには渡しません。
- train は 14 件（根拠あり 10、根拠なし 4）、valid は 7 件です。FAQ が 5 項目しかないので学習データは少なく、同じ例を何回も学習することになります。何回学習するかは、データを複製して増やすのではなく、セクション 4 のエポック数（`num_train_epochs`）で決めます。データのファイルには実例だけを入れ、学習量は学習の設定で持つのが基本です。

**見るところ**: `train=14 valid=7 test=21 gold=21` と、学習例の 1 件（質問と、望ましい出力）が表示されます。質問が train / valid / test のどれかで重複していると、ここでエラーになります。

In [4]:
FAQ = {
    "ask": "質問を 1 回送るには uv run python -m app.ask を使います。",
    "model": "使うモデルは AIE_MODEL という環境変数で切り替えます。",
    "tools": "エージェントのツール一覧は app.agent に --list-tools を付けると表示されます。",
    "trace": "ツールの呼ばれ方は app.agent に --trace を付けて実行すると確認できます。",
    "start_tag": "各回の開始時点のコードは lecNN-start タグから作業ブランチを作って使います。",
}
# モデルに与える回答の根拠情報
# FAQの値だけを取り出し、改行で連結
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

# 根拠あり: (FAQ キー, 質問テンプレート群, 正解の answer に含まれるべき語)
GROUNDED = {
    "ask": (["質問を 1 回だけ送るコマンドは何ですか", "LLM に単発で質問するにはどうしますか", "モデルに一問だけ聞くときのコマンドを教えてください", "第1回目の宿題で使用したコマンドは？"], ["app.ask"]),
    "model": (["使うモデルを変えるにはどうしますか", "モデルの切り替え方法を教えてください", "別のモデルで試したいときは何を設定しますか", "モデルを指定する環境変数は？"], ["AIE_MODEL"]),
    "tools": (["エージェントが使えるツールを一覧で見るには？", "ツールの一覧を表示する方法を教えてください", "モデルに見えているツールを確認したいです", "ツール一覧を出すオプションは？"], ["--list-tools"]),
    "trace": (["ツールがどう呼ばれたか確認するには？", "エージェントの途中の動きを見る方法を教えてください", "どのツールがどの引数で呼ばれたか知りたいです", "ツール呼び出しを表示するオプションは？"], ["--trace"]),
    "start_tag": (["各回の開始時点のコードはどう取り出しますか", "講座の最初の状態からブランチを作るには？", "その回の初期コードから始める方法を教えてください", "開始時点のコードに付いているタグは？"], ["lecNN-start"]),
}
# 根拠なし: FAQ に無いので needs_review=true が正解　→　FAQに答えがない質問を用意する
UNGROUNDED = [
    "app.ask で会話の履歴は残りますか", "Gemini API の料金はいくらですか", "AIE_MODEL に指定できるモデルの一覧は？",
    "--trace の出力をファイルに保存できますか", "ツールを自分で追加するにはどのファイルを直しますか", "Codespaces の無料枠は月に何時間ですか",
    "開始タグを消してしまったらどうなりますか", "AIE_MODEL を設定しないとどのモデルが使われますか",
]
# test 専用の追加質問（train / valid には入れない）
TEST_EXTRA_GROUNDED = {
    "ask": ["質問を 1 つ投げて答えを見るには何を実行しますか", "単発の質問に使う uv run のコマンドを教えてください"],
    "model": ["Gemini のモデル名を変えて実行するには何を使いますか", "実行するモデルはどの環境変数で決まりますか"],
    "tools": ["app.agent で利用できるツールを表示するには？", "エージェントのツール名をまとめて確認する方法は？"],
    "trace": ["エージェントのツール呼び出しの様子を見るオプションを教えてください", "app.agent の実行中にどのツールが使われたか確かめるには？"],
    "start_tag": ["作業ブランチはどのタグから作りますか", "各回のスタート地点のコードを使うにはどうしますか"],
}
TEST_EXTRA_UNGROUNDED = [
    "app.ask の回答を英語にできますか", "--list-tools の表示に引数の説明は出ますか",
    "lecNN-start タグはいつ公開されますか", "講座の修了証はもらえますか",
]
UNGROUNDED_ANSWER = "この FAQ には記載がありません。運営に確認が必要です。"

# 質問と正解を、学習データとして使いやすい共通の形式に変換
def make_example(question: str, answer: str, needs_review: bool) -> dict:
    """モデルに入力できる学習用データを作る関数"""
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

# 4つの質問文を3種類に分ける
# 学習に使用した文章と全く同じ文章で評価してしまうと、単純な暗記でも正解できるため、学習に使用していない質問で評価する
# イメージ
# 　templates = ["質問A","質問B","質問C","質問D",]
# 　↓
# 　train_questions = ["質問A", "質問B"]
# 　valid_questions = ["質問C"]
# 　test_questions = ["質問D"]
def split_templates(templates):
    """
    テンプレートを学習用・検証用・テスト用に分割する関数

    テンプレートの順序を維持し、先頭2件を学習用、
    3件目を検証用、4件目をテスト用に割り当てる。
    言い換えテンプレートが異なるデータセットに混在することを防ぐ。
    """
    # テンプレートの先頭 2 つを train、3 つ目を valid、4 つ目を test に固定（言い換えを跨がせない）
    return templates[:2], templates[2:3], templates[3:4]

# train：学習用のリスト
# valid：検証用のリスト
# test：評価用のリスト
# gold：評価用の正解を保存する辞書
train, valid, test, gold = [], [], [], {}
# 根拠ありの質問を振り分ける
# 　各FAQ項目のキーと値の組み合わせを順番に取り出す
# 　("deadline", (質問リスト, 正解語句リスト))　→ タプルの2つの要素をそれぞれの変数に取り出す(アンパッキング)
for key, (templates, must_include) in GROUNDED.items():
    # 質問文リストをtrain用2件、valid用1件、test用1件に分割する
    tr, va, te = split_templates(templates)
    # train用の質問を1件ずつ取り出す
    # 質問、FAQの回答、確認不要(False)をまとめてtrainへ追加する
    for q in tr: train.append(make_example(q, FAQ[key], False))
    # valid用の質問を1件ずつ取り出す
    # 質問、FAQの回答、確認不要(False)をまとめてvalidへ追加する
    for q in va: valid.append(make_example(q, FAQ[key], False))
    # 元のtest用質問1件と、追加のtest専用質問2件を連結する
    # 連結した質問を1件ずつ取り出す
    for q in te + TEST_EXTRA_GROUNDED[key]:
        # 質問、FAQの回答、確認不要(False)をまとめてtestへ追加する
        # 評価時の正解をgoldに登録する
        # qをキーにして、確認要否と回答に必要な語句を保存する
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}

# 根拠なしの質問を振り分ける
# UNGROUNDEDの質問を番号iと質問文qに分けて取得する
for i, q in enumerate(UNGROUNDED):
    # 質問、根拠なしの場合の定型回答、確認必要(True)をまとめる
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    # i % 4　→　4件ごとに「train、train、valid、test」というパターンを繰り返す
    # iを4で割った余りに応じて、追加先のリストを選ぶ
    # 余り0→train、1→train、2→valid、3→test
    bucket = [train, train, valid, test][i % 4]   # 根拠なしの質問は 4 問ごとに train 2 / valid 1 / test 1 に振る
    # 選ばれたリストに、作成した学習・評価データを追加する
    bucket.append(ex)
    # 選ばれたリストがtestそのものか確認する
    # testの質問だけ、評価用の正解をgoldに追加する
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["確認"]}

# test専用の根拠なし質問を1件ずつ取り出す
for q in TEST_EXTRA_UNGROUNDED:
    # 質問、定型回答、確認必要(True)をtestへ追加する
    # 追加した質問の評価用正解をgoldへ登録する
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["確認"]}

# train は 14 件（根拠あり 10、根拠なし 4）。同じ例を何回学習するかは、データを複製せず、セクション 4 の num_train_epochs で決める
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "質問が train / valid / test のどこかで重複しています。テンプレートを見直してください"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("学習例の 1 件 | 質問:", train[0]["question"], "| 望ましい出力:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

train=14 valid=7 test=21 gold=21
学習例の 1 件 | 質問: 質問を 1 回だけ送るコマンドは何ですか | 望ましい出力: {"answer": "質問を 1 回送るには uv run python -m app.ask を使います。", "needs_review": false}


In [5]:
# 追加
# 学習データを1件取り出す
example = train[0]

# それぞれ何が保存されているか確認する
print("質問:")
print(example["question"])

print("\nモデルに渡すFAQ:")
print(example["faq"])

print("\n学習時の望ましい回答:")
print(example["target"])

# 評価する質問と、対応する正解基準を確認する
question = test[0]["question"]

print("\n評価用の質問:")
print(question)

print("\n採点用の正解:")
print(gold[question])

質問:
質問を 1 回だけ送るコマンドは何ですか

モデルに渡すFAQ:
- 質問を 1 回送るには uv run python -m app.ask を使います。
- 使うモデルは AIE_MODEL という環境変数で切り替えます。
- エージェントのツール一覧は app.agent に --list-tools を付けると表示されます。
- ツールの呼ばれ方は app.agent に --trace を付けて実行すると確認できます。
- 各回の開始時点のコードは lecNN-start タグから作業ブランチを作って使います。

学習時の望ましい回答:
{"answer": "質問を 1 回送るには uv run python -m app.ask を使います。", "needs_review": false}

評価用の質問:
第1回目の宿題で使用したコマンドは？

採点用の正解:
{'needs_review': False, 'must_include': ['app.ask']}


## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の 21 問に答えさせて採点します。

プロンプト（`SYSTEM`）には「必ず JSON 1 行で、`answer` と `needs_review` を返す」と書いてあります。つまり **形式は指示済み** です。このモデルがその指示を守れるかどうかが、学習前の確認点です。

**採点のしかた**（セクション 5 でも同じ関数を使います）

- **形式**: JSON として読め、キーが `answer` と `needs_review` のちょうど 2 つで、型が文字列と真偽値であること。
- **内容**: `needs_review` が正解と一致し、かつ正解の語句（例: 「9 月 30 日」と「23:59」）が `answer` に含まれること。

内容の判定には `needs_review` の一致が必要です。そのため、キーが欠けている出力は、文面が正しくても内容は × になります。文面そのものは、表で自分の目で確かめてください。

`Qwen/Qwen3.5-0.8B` は画像も扱えるモデルですが、`AutoModelForCausalLM` で読み込むとテキスト部分だけが使われます。

**見るところ**

- 1 つ目のセル: `重みのメモリ(GB): 3.01` のように表示されます。モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。
- 2 つ目のセル: 1 行の要約（`format_rate`、`content_rate` など）と、21 問の出力の表。評価には 1 分前後かかります。学習前は、多くの出力で `needs_review` が欠けているはずです。

**出ても問題ない警告**

- `You are sending unauthenticated requests to the HF Hub`: Hugging Face にログインせずにモデルを取りに行っている、という案内です。公開モデルはログインしなくてもダウンロードできます。ログインが要るのは次の場合で、モデルの大きさでは決まりません。
  - 利用規約への同意が必要なモデル（gated model。Llama や Gemma など）や、非公開のリポジトリを使うとき。
  - 自分のモデルやデータを Hub にアップロードするとき。
  - 短時間に大量のリクエストを送り、回数の上限に当たるとき。上限はログインしていないと IP アドレスごとに 5 分あたり 3,000 回のファイル取得（2025 年 9 月時点の公表値。[Hub の rate limits](https://huggingface.co/docs/hub/rate-limits)）で、ログインすると上がります。
  
  このノートブックのモデルは公開されていて同意も不要です。読み込みで送るリクエストは多くても数十回なので、上限には届きません。もし上限に当たって `429 Too Many Requests` が出ても、ライブラリが待ち時間を読み取って自動で再試行します。ログインが必要になったときは、Colab の左側の鍵アイコン（シークレット）に Hugging Face のトークンを `HF_TOKEN` という名前で登録し、このノートブックからのアクセスを許可すると、自動で使われます。
- `... is falling back to its reference PyTorch implementation ...`: このモデルの一部の層（線形注意）に対する高速な実装が入っていないので、遅いが正しい実装で動くという案内です。4 行ほど出ます。

**ここで読んでおくこと**: 表を見て、「どのキーが欠けているか」「文面は FAQ と合っているか」を確認します。この表はセクション 5 で学習後の出力と並べて見ます。出力の原文は `/content/qlora_lab/eval_before.json` にも保存されています（左のファイル一覧から開けます）。

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


# 利用可能な計算デバイスをCUDA → MPS → CPUの優先順位で選択する
# CUDA: NVIDIA GPU、MPS: Apple Silicon GPU、CPU: プロセッサ
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")


# 事前学習済みモデルに対応するTokenizerを読み込む
# TokenizerはテキストとトークンID（整数）を相互変換する
tok = AutoTokenizer.from_pretrained(
    CONFIG["base_model"],          # 読み込むモデルのID
    revision=CONFIG["revision"],   # モデルのバージョン（ブランチ・タグ・コミット等）
)

# パディング用トークンが未設定の場合、文末トークンで代用する
# パディングは長さの異なる入力を同じ長さに揃えるために使用する
if tok.pad_token is None:
    tok.pad_token = tok.eos_token


# 事前学習済みの文章生成モデルを読み込み、指定デバイスへ配置する
# CausalLMは、それまでのトークンをもとに次のトークンを予測する言語モデル
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],          # 読み込むモデルのID
    revision=CONFIG["revision"],   # モデルのバージョン
    dtype=torch.float32,           # 重みを32bit浮動小数点数で読み込む（量子化なし）
).to(DEVICE)                       # モデルを選択したGPUまたはCPUへ移動する

# 生成時に過去のAttention計算結果（KVキャッシュ）を再利用する
# 同じ計算の繰り返しを減らせる一方、キャッシュ用メモリを消費する
model.config.use_cache = True

# モデルのパラメータやバッファのメモリ量を取得してGB単位に換算する
BASE_MEMORY_GB = round(
    model.get_memory_footprint() / 1e9,  # バイト数をGB（10^9バイト）へ変換
    2,                                   # 小数点以下2桁に丸める
)

# 読み込んだモデル名、使用デバイス、メモリ量を表示する
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)


# モデルに与える共通のシステム指示
# FAQを根拠としてJSON形式で回答するよう指示する
# 指示だけでは出力形式を保証できないため、学習前に遵守できるか確認する
SYSTEM = """
あなたは講座リポジトリのコマンド FAQの回答係です。

【回答ルール】
- 与えられたFAQだけを根拠に回答してください。
- 必ずJSON形式の1行で回答してください。
- FAQに記載がない場合は、以下のように回答してください。
  - answer: "確認が必要"
  - needs_review: true

【出力形式】
{
  "answer": "回答内容",
  "needs_review": false
}
"""


def build_messages(example: dict[str, str]) -> list[dict[str, str]]:
    """
    FAQと質問からチャット形式のメッセージを構築する関数

    共通のシステム指示と、FAQ・質問を含むユーザーメッセージを
作成し、モデルへ渡すためのメッセージ一覧として返す。

    Args:
        example (dict[str, str]): FAQと質問を格納した辞書
            faq: 回答の根拠となるFAQ本文
            question: FAQに対する質問文

    Returns:
        list[dict[str, str]]: systemとuserのメッセージを格納したリスト
    """
    # FAQと質問を1つのユーザー入力文字列にまとめる
    # \nは改行、\n\nは空行を挟むことで内容を区切る
    user = f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"

    # チャット形式のメッセージ一覧を返す
    # role: 発言者の役割、content: メッセージの本文
    return [
        {"role": "system", "content": SYSTEM},  # system: モデル全体への共通指示
        {"role": "user", "content": user},      # user: FAQと質問を含む入力
    ]


def generate(model, example: dict[str, str]) -> str:
    """
    FAQと質問をモデルへ入力して回答テキストを生成する関数

    チャットメッセージをモデル専用のプロンプト形式へ変換し、
    トークン化してモデルへ入力する。
    生成されたトークンから入力部分を除き、回答部分だけを文字列で返す。

    Args:
        model: 回答生成に使用する事前学習済み言語モデル
        example (dict[str, str]): FAQと質問を格納した辞書
            faq: 回答の根拠となるFAQ本文
            question: FAQに対する質問文

    Returns:
        str: モデルが生成した回答テキスト
    """
    # チャットメッセージをモデル固有のプロンプト形式に変換する
    # apply_chat_template()は、チャットメッセージを、対象モデルが学習時に使用した会話形式に合わせて整形するメソッド
    # 言語モデルによって、発言者や発言の区切りを表す特殊トークンが異なるため
    prompt = tok.apply_chat_template(
        build_messages(example),        # 変換対象のメッセージ一覧（system・user）
        tokenize=False,                 # トークンIDに変換せず文字列として返す
        add_generation_prompt=True,     # assistantの回答開始を示す部分を末尾に追加する
        enable_thinking=False,          # 対応モデルのThinkingモードを無効にする
    )

    # プロンプト文字列をトークンIDへ変換する
    # .to(model.device)でモデルと同じデバイスに入力Tensorを配置する
    inputs = tok(
        prompt,                         # トークン化するプロンプト文字列
        return_tensors="pt",            # PyTorchのTensor形式で返す
    ).to(model.device)

    # 推論時は勾配計算を無効にして、不要な計算とメモリ消費を抑える
    with torch.no_grad():
        # 入力トークンに続く新しいトークンを順番に生成する
        out = model.generate(
            **inputs,                                  # input_idsなどの入力をキーワード引数として展開
            max_new_tokens=CONFIG["max_new_tokens"],   # 新しく生成するトークン数の上限
            pad_token_id=tok.pad_token_id,             # パディングに使用するトークンID
            **CONFIG["gen"],                           # temperatureなどの生成設定を展開
        )

    # 入力トークン数を取得する
    # shape[1]はTensorの2番目の次元（トークン列の長さ）
    input_length = inputs["input_ids"].shape[1]

    # 生成結果から入力部分を除き、回答トークンだけを取り出す
    # out[0]はバッチ内の最初の生成結果
    # [input_length:]は入力部分を飛ばして生成部分を取得するスライス
    generated_ids = out[0][input_length:]

    # 回答トークンを人間が読める文字列に戻す
    answer = tok.decode(
        generated_ids,                 # 文字列へ変換するトークンID
        skip_special_tokens=True,      # 文末記号などの特殊トークンを除外する
    )

    # 文字列の前後にある空白や改行を削除して返す
    return answer.strip()

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [7]:
import re
import pandas as pd

# pandasの表示設定を変更する
pd.set_option(
    "display.max_colwidth",  # DataFrameの各列に表示できる文字列の最大幅
    None,                    # 文字列を幅によって省略しない
)


def check_format(text: str):
    """
    モデルの出力が指定されたJSON形式を満たすか検証する関数

    JSONとして解析できることに加え、必須キーの存在、
    余分なキーの有無、各値のデータ型を検証する。

    Args:
        text (str): モデルが生成した回答文字列

    Returns:
        tuple: 検証結果、失敗理由、解析済みのJSONオブジェクト
            - 検証結果: 形式が正しければTrue
            - 失敗理由: 正常時は空文字列
            - JSONオブジェクト: 解析できなかった場合はNone
    """
    # json.loads(): JSON文字列をPythonのオブジェクトへ変換するAPI
    # JSONとして解析できない場合も評価を継続するため、例外を捕捉する
    try:
        obj = json.loads(text)  # 入力: JSON文字列、出力: dict・list・strなど
    except Exception as e:
        # type(e).__name__で例外の種類を取得し、失敗理由として記録する
        return False, f"json_error: {type(e).__name__}", None

    # JSONの最上位要素がオブジェクト（Pythonのdict）か確認する
    # JSONの配列や文字列も解析可能なため、別途確認が必要
    if not isinstance(obj, dict):
        return False, "not_object", None

    # 許可されていないキーが含まれていないか確認する
    # setの差集合（-）で、許可キー以外の要素を取り出す
    if set(obj) - {"answer", "needs_review"}:
        return False, "extra_keys", obj

    # 回答に必要な2つのキーが両方存在するか確認する
    if "answer" not in obj or "needs_review" not in obj:
        return False, "missing_key", obj

    # JSONの値が指定した型と一致するか確認する
    # answerは文字列、needs_reviewは真偽値である必要がある
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool):
        return False, "wrong_type", obj

    # すべての形式チェックに合格
    return True, "", obj


def check_content(obj, gold_entry):
    """
    モデルの回答内容が正解データの条件を満たすか判定する関数

    needs_reviewの一致と、回答に含めるべき語句の存在を確認する。
    完全一致ではなく、指定された条件による簡易的な採点を行う。

    Args:
        obj (dict | None): JSONから解析した回答オブジェクト
        gold_entry (dict): 正解データ
            needs_review: 期待する真偽値
            must_include: 回答に含める必要がある語句の一覧

    Returns:
        bool: すべての条件を満たす場合はTrue
    """
    # JSONを解析できなかった場合、内容を採点できない
    if obj is None:
        return False

    # 確認が必要かどうかの判定が正解と一致するか確認する
    # dict.get()はキーが存在しない場合、既定値のNoneを返す
    if obj.get("needs_review") != gold_entry["needs_review"]:
        return False

    # all(): すべての条件がTrueの場合だけTrueを返すAPI
    # 正解データに指定された必須語句が、回答にすべて含まれるか確認する
    return all(
        w in obj.get("answer", "")  # w: 必須語句、answer: モデルの回答文字列
        for w in gold_entry["must_include"]
    )


def mark(ok: bool) -> str:
    """
    真偽値の評価結果を表示用の記号に変換する関数

    Args:
        ok (bool): 評価結果

    Returns:
        str: 成功の場合は○、失敗の場合は×
    """
    return "○" if ok else "×"


def evaluate(model, label: str):
    """
    テストデータに対するモデルの回答を生成・採点・保存する関数

    各質問に回答を生成し、JSON形式と回答内容を別々に評価する。
    評価結果を集計してJSONファイルへ保存し、
    全体の評価指標と質問ごとの詳細結果を返す。

    Args:
        model: 評価対象の言語モデル
        label (str): 評価を識別するラベル（例: before、after）

    Returns:
        tuple: 全体の評価結果と質問ごとの評価結果
            - summary (dict): 正解率、評価件数、実行時間など
            - rows (list[dict]): 各質問の出力と採点結果
    """
    # 各質問の評価結果を格納するリスト
    rows = []
    # time.time(): 現在時刻をUNIX時間（秒）で取得するAPI
    # 評価開始時刻を保存し、終了時との差分から処理時間を計測する
    t0 = time.time()
    # テストデータに含まれる質問を1件ずつ評価する
    for ex in test:
        # FAQと質問をモデルへ入力し、回答文字列を生成する
        raw = generate(model, ex)
        # 回答が指定したJSON形式を満たしているか確認する
        ok_fmt, reason, obj = check_format(raw)
        # 形式に問題があっても、解析可能な内容は別途採点する
        # 形式の正しさと回答内容の正しさを切り分けるため
        loose = obj
        # JSON形式が不正な場合、回答内からJSON部分を抽出する
        if not ok_fmt:
            # re.search(): 正規表現に一致する最初の部分を検索するAPI
            # 回答全体が正しいJSONでなくても、{...}の部分を抽出する
            m = re.search(
                r"\{.*\}",  # {から}までの文字列を検索するパターン
                raw,        # 検索対象のモデル出力
                re.S,       # .が改行にも一致するようにする（DOTALL）
            )
            # 抽出した文字列をJSONとして解析する
            try:
                loose = json.loads(m.group(0)) if m else None
            except Exception:
                loose = None
        # 正解データと照合して回答内容を評価する
        # 形式が不正でも辞書として解析できた場合は内容を採点する
        ok_content = check_content(
            loose if isinstance(loose, dict) else None,  # 採点対象の回答
            gold[ex["question"]],                         # 質問に対応する正解データ
        )
        # 質問ごとの評価結果を保存する
        # 形式・内容・両方合格を区別し、失敗原因を分析できるようにする
        rows.append({
            "question": ex["question"],                                 # 評価対象の質問
            "raw": raw,                                                 # モデルの出力原文
            "format_ok": ok_fmt,                                        # JSON形式の合否
            "format_reason": reason,                                    # 形式が不正な理由
            "content_ok": ok_content,                                   # 回答内容の合否
            "both_ok": ok_fmt and ok_content,                           # 形式と内容の両方が合格したか
            "gold_needs_review": gold[ex["question"]]["needs_review"],  # 正解の確認要否
        })
    # 評価対象の質問数を取得する
    n = len(rows)
    # 評価全体の指標を計算する
    # 正解率 = 合格件数 ÷ 評価件数
    # boolはPythonではTrue=1、False=0として合計できる
    summary = {
        "label": label,                                       # 評価の識別名
        "n": n,                                               # 評価した質問数
        "format_rate": sum(r["format_ok"] for r in rows) / n, # JSON形式の合格率
        "content_rate": sum(r["content_ok"] for r in rows) / n,  # 回答内容の合格率
        "both_rate": sum(r["both_ok"] for r in rows) / n,     # 形式・内容の両方の合格率
        "seconds": round(time.time() - t0, 1),               # 評価全体の所要時間（秒）
    }
    # 評価結果をJSONファイルとして保存する
    # 後から学習前後の結果を比較・分析できるようにする
    # pathlib.Path.write_text(): 文字列を指定したファイルへ書き込むAPI
    (WORK / f"eval_{label}.json").write_text(
        json.dumps(
            {"summary": summary, "rows": rows},  # 保存する評価データ
            ensure_ascii=False,                  # 日本語をUnicodeエスケープせず保存する
            indent=2,                            # 2スペースでインデントして整形する
        ),
        encoding="utf-8",                         # ファイルの文字エンコーディング
    )
    # 評価結果の要約をNotebookの出力へ表示する
    print(json.dumps(summary, ensure_ascii=False))
    # 全体の集計結果と質問ごとの詳細結果を返す
    return summary, rows

# ファインチューニング前のモデルを評価する
# 後から学習後の結果と比較するため、beforeとして保存する
before_summary, before_rows = evaluate(model, "before")

# 質問ごとの評価結果を表形式で表示する
# pandas.DataFrame(): 辞書のリストを行と列を持つ表データへ変換するAPI
# display(): Jupyter Notebook上に表を表示する
display(
    pd.DataFrame([
        {
            "質問": r["question"],
            "正解の needs_review": r["gold_needs_review"],
            "学習前の出力": r["raw"],
            "形式": mark(r["format_ok"]),
            "形式の失敗理由": r["format_reason"],
            "内容": mark(r["content_ok"]),
        }
        for r in before_rows
    ])
)


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 21, "format_rate": 0.0, "content_rate": 0.0, "both_rate": 0.0, "seconds": 55.2}


,質問,正解の needs_review,学習前の出力,形式,形式の失敗理由,内容
0,第1回目の宿題で使用したコマンドは？,False,"{\n ""answer"": ""第1回目の宿題で使用したコマンドは uv run python -m app.ask です。""\n}",×,missing_key,×
1,質問を 1 つ投げて答えを見るには何を実行しますか,False,"{\n ""answer"": ""uv run python -m app.ask を使います。""\n}",×,missing_key,×
2,単発の質問に使う uv run のコマンドを教えてください,False,"{\n ""answer"": ""uv run python -m app.ask を使います。""\n}",×,missing_key,×
3,モデルを指定する環境変数は？,False,"{\n ""answer"": ""AIE_MODEL""\n}",×,missing_key,×
4,Gemini のモデル名を変えて実行するには何を使いますか,False,"{\n ""answer"": ""Gemini のモデル名を変えて実行するには、uv run python -m app.ask を使います。使うモデルは AIE_MODEL という環境変数で切り替えます。""\n}",×,missing_key,×
5,実行するモデルはどの環境変数で決まりますか,False,"{\n ""answer"": ""AIE_MODEL""\n}",×,missing_key,×
6,ツール一覧を出すオプションは？,False,"{\n ""answer"": ""app.agent に --list-tools を付けると表示されます。""\n}",×,missing_key,×
7,app.agent で利用できるツールを表示するには？,False,"{\n ""answer"": ""app.agent でツールを表示するには、--list-tools を付けます。""\n}",×,missing_key,×
8,エージェントのツール名をまとめて確認する方法は？,False,"{\n ""answer"": ""app.agent に --list-tools を付けると表示されます。""\n}",×,missing_key,×
9,ツール呼び出しを表示するオプションは？,False,"{\n ""answer"": ""uv run python -m app.ask を使います。使うモデルは AIE_MODEL という環境変数で切り替えます。エージェントのツール一覧は app.agent に --list-tools を付けると表示されます。ツール呼び出しは app.agent に --trace を付けて実行すると確認できます。""\n}",×,missing_key,×


## 4. LoRA 学習（T4 で 4〜5 分）

### LoRA と QLoRA の違い

- **LoRA**: 基盤モデルの重みは固定し、一部の層に追加した小さな行列（adapter）だけを学習します。
- **QLoRA**: 基盤モデルの重みを 4bit に量子化して保持し、その上で同じように adapter だけを学習します。adapter 自体は量子化しません。

QLoRA の効果は、基盤モデルの重みを保持するメモリが減ることです。16bit の約 4 分の 1 になるので、そのままでは GPU に載らない大きさのモデルでも学習できます（70 億パラメータなら 16bit で約 14GB、4bit で約 3.5GB）。

| 観点 | LoRA（量子化なし） | QLoRA（4bit） |
| --- | --- | --- |
| 重みのメモリ | 大きい | 小さい（16bit の約 1/4） |
| 基盤モデルの出力 | 元のまま | 量子化の誤差で変わることがある。影響の大きさはモデルによって違う |
| 学習の速さ | 基準 | 速くはならない。4bit の重みを計算のたびに戻すので、遅くなることが多い |
| 依存ライブラリ | peft | peft に加えて bitsandbytes（CUDA の GPU が前提） |

QLoRA を実装するときの注意:

- 読み込み時に `BitsAndBytesConfig(load_in_4bit=True, ...)` を渡し、LoRA を付ける前に `prepare_model_for_kbit_training(model)` を通します。
- 保持精度（4bit）と計算精度（`bnb_4bit_compute_dtype`。16bit など）は別の設定です。
- 学習前後の比較は、同じ量子化条件の基盤モデルで行います。学習前を量子化なし、学習後を 4bit で測ると、量子化の影響と学習の影響が混ざります。
- adapter は学習時の基盤モデルに合わせて作られます。配布や再利用のときは、基盤モデルの ID・revision と量子化設定を一緒に記録します。
- 4bit での学習が勧められないモデルもあります。Qwen3.5 は量子化の誤差が通常より大きいとして、4bit 学習を勧めていない[学習ライブラリのガイド](https://unsloth.ai/docs/models/qwen3.5/fine-tune)があります。

この演習のモデルは量子化なしでも T4 に載るので、LoRA で学習します。モデルが GPU に載るなら LoRA、載らないなら QLoRA かより小さいモデル、という順で考えます。4bit で読み込んだ場合のメモリと出力の違いは、セクション 8（発展）で確かめられます。

### 学習の実行

**やること**: train の 14 件を chat 形式（system / user / assistant）に直し、LoRA を付けて 20 エポック（40 ステップ）学習します。学習した adapter は `adapter/` に保存します。

- `target_modules` は LoRA を差し込む層です。注意機構の射影（`q_proj` など）と MLP の射影（`gate_proj` など）を指定しています。このモデルは 24 層のうち 18 層が線形注意で、`q_proj` などを持つのは残りの 6 層だけです。MLP は全層にあります。
- バッチ 4 × 勾配累積 2 = 1 ステップ 8 件。train は 14 件なので 1 エポックは 2 ステップ（8 件と 6 件）で、20 エポックで 40 ステップです。各例をちょうど 20 回学習します。学習率の warmup と減衰（cosine）はこの 40 ステップに対して効きます。
- 学習後のモデルは変数 `tuned` に入ります。LoRA は基盤モデルの層に差し込まれるので、元の変数 `model` で生成しても adapter 込みの出力になります。学習前の出力が必要なときは、セクション 6 のように adapter を一時的に無効にします。

**見るところ**

- `trainable params: ... || all params: ... || trainable%: ...`: 学習するパラメータが全体の数 % だけであることが LoRA の特徴です。
- `Training Loss` の表（10 ステップごと）: 1.5 前後から始まり、40 ステップで 0.01〜0.02 まで下がります。下がらないまま終わった場合は、設定を変えていないか確かめてください。
- `学習時間: ... 秒 | ステップ数: 40 | エポック: 20.0`: T4 で 240〜290 秒です。ステップ数とエポックは、学習データ 14 件 × 20 周が上の計算どおりに実行されたことの確認です。

**出ても問題ない警告**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` は、終端トークンの設定を tokenizer に合わせたという案内です。

**うまくいかないとき**: 学習が途中で止まった、または学習できなかった場合は、下の 4b を読んでください。

In [8]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset


# モデルの学習形式に変換
def to_chat(example):
    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
    ,target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(
    output_dir=str(WORK / "ckpt"),
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=CONFIG["lr"],
    lr_scheduler_type="cosine",
    warmup_steps=CONFIG["warmup_steps"],
    logging_steps=10,
    save_steps=10,
    save_total_limit=2,
    fp16=USE_AMP,
    max_length=CONFIG["max_seq_len"],
    packing=False,
    report_to="none",
    seed=CONFIG["seed"]
)
trainer = SFTTrainer(
    model=model,
    args=sft,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    peft_config=lora,
    processing_class=tok
)


# ============================================================
# ① 実験ごとの保存先を作成
# ============================================================
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path
# 実行日時を使って実験IDを作る。
# 同じ設定で再学習しても、過去の実験を上書きしないため。
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")

# Google Drive内に、実験ごとのフォルダを作成する。
if USE_DRIVE:
    RUN_DIR = Path(CONFIG["drive_dir"]) / run_id
    RUN_DIR.mkdir(parents=True, exist_ok=False)

# 学習を実行
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
# 学習実行にかかった時間を記録
training_seconds = time.time() - t0
# 最後に記録されたTraining Lossを取得
loss_logs = [
    entry
    for entry in trainer.state.log_history
    if "loss" in entry
]
final_loss = loss_logs[-1]["loss"]
print(f"最終Training Loss: {final_loss}")
print(f"学習時間: {training_seconds:.1f} 秒")

# 学習終了時点で結果を記録する。
# 評価前にランタイムが切れても、Driveに実測値を残すため。
training_metrics = {
    "final_loss": final_loss,
    "training_seconds": round(training_seconds, 1),
    "steps": trainer.state.global_step,
}

if USE_DRIVE:
    # 学習条件も一緒に保存し、後から実験を識別できるようにする。
    training_record = {
        "config": {
            k: v for k, v in CONFIG.items()
            if k not in ("work_dir", "drive_dir")
        },
        "training": training_metrics,
    }

    (RUN_DIR / "training_metrics.json").write_text(
        json.dumps(training_record, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print("学習記録の保存先:", RUN_DIR / "training_metrics.json")

ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    # import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
    # 実験ごとにアダプターを保存
    shutil.copytree(
        ADAPTER_DIR,
        RUN_DIR / "adapter",
        dirs_exist_ok=True,
    )
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

Tokenizing train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

... (2 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,1.995886
20,0.128449
30,0.030193
40,0.017329


学習時間: 245 秒 | ステップ数: 40 | エポック: 20.0
最終Training Loss: 0.01732919067144394
学習時間: 244.7 秒
学習記録の保存先: /content/drive/MyDrive/aie2026/runs/20261127T125056_202095Z/training_metrics.json


### 4b. 学習が止まった・できなかった場合

**途中で止まった（ランタイムの切断など）**: チェックポイントは 10 ステップごとに `/content/qlora_lab/ckpt/` に保存されています。ファイルが残っていれば、セクション 0〜3 を実行し直してから、上の学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから 40 ステップ（20 エポック）まで続きを学習します。ランタイムごと消えてファイルが無い場合は、学習をやり直すか、下の方法に切り替えます。

**学習できなかった（GPU が確保できない、時間内に終わらない）**: 講師が同じノートブックを T4 で実行した結果（学習後の出力とスコア）を使って、比較と考察を進めます。セクション 5 以降のセルは学習後のモデル `tuned` が無いと動かないので、講師の結果を見ながら説明を読んでください。提出するときは、セクション 7 の `executed_training_myself` を `False` にし、本人が実行できた範囲を明記します。

## 5. 学習後評価（学習前と同じ test・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 21 問・同じ生成設定で評価し、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**

- 比較表: `format_rate`（形式遵守率）と `content_rate`（内容正解率）が学習前からどう変わったか。
- `退行`: 学習前は形式も内容も通っていたのに、学習後に落ちた質問です。空のリストが期待値です。
- 質問ごとの表: 残った失敗を読み、どの型かを見分けてください。
  - 根拠なしの質問（正解の `needs_review` が `True`）に、FAQ の別項目を当てはめて断定した。
  - 根拠ありの質問（正解が `False`）に、「記載がありません」と確認要求で逃げた。

学習後の回答は、教師データの文（FAQ の原文）をほぼそのまま返す形になります。形式と文面は揃いますが、「FAQ に根拠があるか」の判断まで学習できたとは限りません。

seed を固定しても、GPU 上の学習は完全には再現しません。形式遵守率と内容正解率の割合はほぼ同じでも、どの質問が残るかは実行ごとに変わることがあります。隣の人と「残った質問」が違っても異常ではないので、失敗の型で比べてください。

In [9]:
# ① 学習後のモデルを評価する
# evaluate(モデル, 評価名)：21問の回答を生成・採点する自作関数。
# 戻り値は全体の集計結果（summary）と質問ごとの結果（rows）。
# "after"を指定すると WORK / "eval_after.json" に保存される（同名なら上書き）。
after_summary, after_rows = evaluate(tuned, "after")


# ② 学習前後の評価結果を比較表にする
# DataFrame()：学習前後の辞書を2行の表に変換する。
# set_index("label")：行番号の0・1を、label列の値before・afterに置き換える。
# 最後の[...]で表示する5列を選ぶ。secondsは学習時間ではなく評価時間。
compare = pd.DataFrame(
    [before_summary, after_summary]
).set_index("label")[
    ["n", "format_rate", "content_rate", "both_rate", "seconds"]
]
display(compare)


# ③ 学習前は合格、学習後は不合格になった質問（退行）を抽出する
# zip()で学習前後の同じ位置の結果を組にする（質問順が同じことが前提）。
# both_okは形式・内容の両方に合格したかを示す。
regressed = [
    b["question"]
    for b, a in zip(before_rows, after_rows)
    if b["both_ok"] and not a["both_ok"]
]
print("退行:", regressed)  # []なら該当する質問なし


# ④ 21問それぞれの回答と採点結果を比較表にする
# zip()で同じ質問の学習前（b）と学習後（a）を対応付ける。
# rawはモデルの回答原文、mark()はTrue/Falseを○/×に変換する自作関数。
# 各質問を1行の辞書にまとめ、DataFrame()で表に変換する。
detail = pd.DataFrame([
    {
        "質問": b["question"],
        "正解の needs_review": b["gold_needs_review"],
        "学習前の出力": b["raw"],
        "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
        "学習後の出力": a["raw"],
        "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}",
    }
    for b, a in zip(before_rows, after_rows)
])
display(detail)

{"label": "after", "n": 21, "format_rate": 1.0, "content_rate": 0.7142857142857143, "both_rate": 0.7142857142857143, "seconds": 40.8}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,21,0.0,0.000000,0.000000,55.2
after,21,1.0,0.714286,0.714286,40.8


退行: []


,質問,正解の needs_review,学習前の出力,学習前 形式/内容,学習後の出力,学習後 形式/内容
0,第1回目の宿題で使用したコマンドは？,False,"{\n ""answer"": ""第1回目の宿題で使用したコマンドは uv run python -m app.ask です。""\n}",×/×,"{""answer"": ""各回の開始時点のコードは lecNN-start タグから作業ブランチを作って使います。"", ""needs_review"": false}",○/×
1,質問を 1 つ投げて答えを見るには何を実行しますか,False,"{\n ""answer"": ""uv run python -m app.ask を使います。""\n}",×/×,"{""answer"": ""ツールの呼ばれ方は app.agent に --trace を付けて実行すると確認できます。"", ""needs_review"": false}",○/×
2,単発の質問に使う uv run のコマンドを教えてください,False,"{\n ""answer"": ""uv run python -m app.ask を使います。""\n}",×/×,"{""answer"": ""質問を 1 回送るには uv run python -m app.ask を使います。"", ""needs_review"": false}",○/○
3,モデルを指定する環境変数は？,False,"{\n ""answer"": ""AIE_MODEL""\n}",×/×,"{""answer"": ""使うモデルは AIE_MODEL という環境変数で切り替えます。"", ""needs_review"": false}",○/○
4,Gemini のモデル名を変えて実行するには何を使いますか,False,"{\n ""answer"": ""Gemini のモデル名を変えて実行するには、uv run python -m app.ask を使います。使うモデルは AIE_MODEL という環境変数で切り替えます。""\n}",×/×,"{""answer"": ""使うモデルは AIE_MODEL という環境変数で切り替えます。"", ""needs_review"": false}",○/○
5,実行するモデルはどの環境変数で決まりますか,False,"{\n ""answer"": ""AIE_MODEL""\n}",×/×,"{""answer"": ""使うモデルは AIE_MODEL という環境変数で切り替えます。"", ""needs_review"": false}",○/○
6,ツール一覧を出すオプションは？,False,"{\n ""answer"": ""app.agent に --list-tools を付けると表示されます。""\n}",×/×,"{""answer"": ""エージェントのツール一覧は app.agent に --list-tools を付けると表示されます。"", ""needs_review"": false}",○/○
7,app.agent で利用できるツールを表示するには？,False,"{\n ""answer"": ""app.agent でツールを表示するには、--list-tools を付けます。""\n}",×/×,"{""answer"": ""エージェントのツール一覧は app.agent に --list-tools を付けると表示されます。"", ""needs_review"": false}",○/○
8,エージェントのツール名をまとめて確認する方法は？,False,"{\n ""answer"": ""app.agent に --list-tools を付けると表示されます。""\n}",×/×,"{""answer"": ""エージェントのツール一覧は app.agent に --list-tools を付けると表示されます。"", ""needs_review"": false}",○/○
9,ツール呼び出しを表示するオプションは？,False,"{\n ""answer"": ""uv run python -m app.ask を使います。使うモデルは AIE_MODEL という環境変数で切り替えます。エージェントのツール一覧は app.agent に --list-tools を付けると表示されます。ツール呼び出しは app.agent に --trace を付けて実行すると確認できます。""\n}",×/×,"{""answer"": ""ツールの呼ばれ方は app.agent に --trace を付けて実行すると確認できます。"", ""needs_review"": false}",○/○


## 6. 知識を問う質問（形式を学んでも知識は保証されない）

**やること**: 形式の実験とは別に、FAQ にも学習データにも無い事実を 3 問尋ねます（対照として FAQ にある質問を 1 問加えます）。望ましい応答は「確認が必要（`needs_review: true`）」です。

応答を 3 つに分類します。

| 分類 | 意味 |
| --- | --- |
| `format_error` | JSON の形式を守れていない（学習前に多い） |
| `needs_review` | 「記載がありません。確認が必要です」と返した（望ましい応答） |
| `asserted` | `needs_review: false` で言い切った。FAQ に無い質問に対してなら、根拠のない断定（創作）です |

**見るところ**: 学習前は形式違反になりやすく、学習後は「確認が必要」と「断定」の両方が出ることがあります。出力の原文も表に出すので、断定した場合に何を根拠にしたように見えるか（FAQ の別項目を流用しているか）を読んでください。「Fine-Tuning では知識は絶対に直らない」と先に結論にせず、今回のデータでの結果を報告します。

学習前の出力は、adapter を一時的に無効にして取っています（`tuned.disable_adapter()`）。



メモ

- `needs_review`: 人間による確認が必要かを表す真偽値（trueまたはfalse）
    - false: FAQに回答の根拠があると判断し、回答する
    - true: FAQだけでは回答できないと判断し、人間に確認を求める
- `asserted`: `needs_review=false`で断定して回答した
    - 断定が正しいかは別途判断がいる
    - assertedだから正解、needs_reviewだから不正解、という意味ではありません。
    - FAQにない質問に対してassertedなら、「根拠のない断定」をしている可能性があります。
    - 反対に、FAQに回答が書いてあるのにneeds_reviewなら、答えられる質問から逃げてしまった可能性がある(「根拠ありの質問に確認要求で逃げた 」)

In [10]:
# - 架空（かくうFAQ に無いだけでなく、そもそも存在しないものを聞く質問。答えがどこにも無い。FAQの文と似せて作り、取り違えて断定するかを見るひっかけにする（例：元の教材の「最終課題の締切」、今回の「app.ask に --list-tools を付けると何が出ますか」）。
# - 対照（たいしょう）：答えが分かっていて、ほかの結果と比べる基準にする質問。ここでは FAQに答えがある質問を置き、答えれば正解。対照があると、ほかの質問で確認を求めたのが「見分けた」のか「全部逃げた」のかを区別できる。
# - 対照 は「照らし合わせて比べる相手」という意味です。薬の試験で、薬を飲んだ人たちと比べるために、飲まない人たち（対照群）を用意するのと同じ考え方です。
# - 対照（たいしょう）：「何でも『確認が必要』と答えているだけ」を見破るために、わざとまぜる、FAQに答えがある質問。FAQ に無い質問で確認を求め、対照の質問には答えたなら、本当に見分けられていると言える。


# KNOWLEDGE_QUESTIONS = [
#     "この講座の最終課題の締切はいつですか",         # FAQ に無い（架空）
#     "担当教員の名前は何ですか",                     # FAQ に無い
#     "次回の講義のテーマは何ですか",                 # FAQ に無い
#     "課題の提出期限はいつですか",                   # FAQ にある（対照）
# ]
KNOWLEDGE_QUESTIONS = [
    "app.judge は何をするコマンドですか",          # FAQ に無い（第 4 回で使ったが FAQ には書いていない）
    "この講座の第 7 回のテーマは何ですか",         # FAQ に無い
    "評価セットは何件で作りますか",                # FAQ に無い
    "モデルを指定する環境変数は？",                # FAQ にある（対照。test にある質問と同じ文にする）
]
def classify(raw):
    # check_format()でモデルの出力rawを検証し、形式の合否(ok)と解析済みJSON(obj)を取得する。
    # _ は今回使用しない戻り値を受け取る変数。
    ok, _, obj = check_format(raw)
    # JSON形式が不正、または解析結果がない場合は、形式エラーと分類して終了する。
    if not ok or obj is None: return "format_error"
    # needs_reviewがTrueなら「確認が必要」、Falseなら「断定した回答」に分類する。
    return "needs_review" if obj["needs_review"] else "asserted"

knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # LoRA は基盤モデルの層に差し込まれるので、学習後は `model` で生成しても adapter 込みになる。学習前の出力は adapter を一時的に無効にして取る
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append(
        {
            "質問": q,
            "FAQ にある": q in gold,   # 質問文qが、テスト用の正解辞書goldのキーとして登録されているかを調べる
            "学習前の分類": classify(before_raw),
            "学習前の出力": before_raw,
            "学習後の分類": classify(after_raw),
            "学習後の出力": after_raw
        }
    )
display(pd.DataFrame(knowledge_rows))

,質問,FAQ にある,学習前の分類,学習前の出力,学習後の分類,学習後の出力
0,app.judge は何をするコマンドですか,False,format_error,"{\n ""answer"": ""app.judge は app.agent に --trace を付けて実行すると確認できます。""\n}",needs_review,"{""answer"": ""この FAQ には記載がありません。運営に確認が必要です。"", ""needs_review"": true}"
1,この講座の第 7 回のテーマは何ですか,False,asserted,"{\n ""answer"": ""この講座の第 7 回のテーマは「LLM の生成モデルの学習と評価」です。"",\n ""needs_review"": false\n}",asserted,"{""answer"": ""この講座の開始時点のコードは lecNN-start タグから作業ブランチを作って使います。"", ""needs_review"": false}"
2,評価セットは何件で作りますか,False,format_error,"{""answer"": ""FAQに記載がありません。""}",needs_review,"{""answer"": ""この FAQ には記載がありません。運営に確認が必要です。"", ""needs_review"": true}"
3,モデルを指定する環境変数は？,True,format_error,"{\n ""answer"": ""AIE_MODEL""\n}",asserted,"{""answer"": ""使うモデルは AIE_MODEL という環境変数で切り替えます。"", ""needs_review"": false}"


## 7. 比較表と提出

**やること**: 設定、ライブラリの版、学習前後の要約、退行した質問を 1 つの `result.json` にまとめます。

**提出するもの**

- `result.json`、`eval_before.json`、`eval_after.json`（`/content/qlora_lab` にあります。左のファイル一覧で右クリック → ダウンロード。`USE_DRIVE = True` なら Drive にもコピーされます）
- セクション 5 の比較表と、残った失敗の説明（どの型の失敗か）
- セクション 6 の結果（確認要求に変わった応答、断定が残った応答）
- 本人が学習を実行した範囲。講師の結果を使った場合は `executed_training_myself` を `False` にしてください。

In [11]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "training": training_metrics, # 実験結果を入れる
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
# if USE_DRIVE:
#     import shutil
#     for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
#         shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
#     print("Drive に保存しました:", CONFIG["drive_dir"])
# 学習結果と評価結果を、今回の実験専用フォルダへ保存する。
if USE_DRIVE:
    for name in [
        "result.json",
        "eval_before.json",
        "eval_after.json",
        "test.jsonl",
        "gold.json",
    ]:
        shutil.copy2(WORK / name, RUN_DIR / name)

    print("今回の実験結果の保存先:", RUN_DIR)

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 20,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.14.1+cu130"
  },
  "method": "LoRA（量子化なし）",
  "training": {
    "final_loss": 0.01732919067144394,
    "training_seconds": 244.7,
    "steps": 40
  },
  "before": {
    "label": "before",
    "n": 21,
    "format_rate": 0.0,
    "content_rate": 0.0,
    "both_rate": 0.0,
    "seconds": 55.2
  },
  "after": {
    "label": "after",
    "n": 21,
    "format_rate": 1.0,
    "content_rate": 0.7142857142857143,
    "both_rate": 0.7142857142857143,
    "seconds": 40.8
  },
  "regressed": [],
  "executed_training_myself": true
}
今回の実験結果の保存先: /content/drive/M

## 8. 発展: 4bit 量子化で読み込み、メモリと出力を比べる（任意）

**やること**: 同じモデルを 4bit（NF4）で読み込み、重みのメモリと、学習前の出力がどう変わるかを見ます。QLoRA は、この 4bit の基盤モデルに LoRA を付けて学習する方法です。ここでは学習はしません。

- bitsandbytes は CUDA の GPU が前提なので、GPU ランタイムでだけ動きます。導入に 1 分ほどかかります。
- 4bit の読み込みは `device_map="auto"` で GPU に配置します（bitsandbytes の読み込み方式に合わせています）。
- 4bit 側で学習まで試す場合は、`prepare_model_for_kbit_training(model_4bit)` を通してからセクション 4 と同じ手順で学習し、学習前後とも 4bit の基盤モデルで評価します。

**見るところ**

- 重みのメモリ: 32bit の約 3GB に対して 4bit は 1GB 未満になります。
- 比較表と質問ごとの表: 学習していないので、32bit の学習前（`before`）との差は量子化の影響です。良くなる方向に変わる質問もあれば、`needs_review` が `"false"`（文字列）になって型検査で落ちる、根拠なしの質問に「はい」「なし」と断定する、といった出力も見られます。方向が一定でないことを確かめてください。

In [12]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # 保持精度: 4bit（NF4）
    bnb_4bit_compute_dtype=torch.float16, # 計算精度: 16bit（T4 は bf16 に対応していない）
    bnb_4bit_use_double_quant=True
)
model_4bit = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto"
)
print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# 学習はしていないので、量子化なしの学習前（before）との差が量子化の影響
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame(
    [before_summary, q_summary]
).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([
    {
        "質問": r["question"],
        "正解の needs_review": r["gold_needs_review"],
        "4bit・学習前の出力": r["raw"],
        "形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}",
        "形式の失敗理由": r["format_reason"]
    } for r in q_rows
]))

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

重みのメモリ(GB) | 量子化なし（32bit）: 3.01 | 4bit: 0.76
{"label": "before_4bit", "n": 21, "format_rate": 0.5714285714285714, "content_rate": 0.5714285714285714, "both_rate": 0.5714285714285714, "seconds": 48.7}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,21,0.000000,0.000000,0.000000,55.2
before_4bit,21,0.571429,0.571429,0.571429,48.7


,質問,正解の needs_review,4bit・学習前の出力,形式/内容,形式の失敗理由
0,第1回目の宿題で使用したコマンドは？,False,"{\n ""answer"": ""uv run python -m app.ask を使います。"",\n ""needs_review"": false\n}",○/○,
1,質問を 1 つ投げて答えを見るには何を実行しますか,False,"{\n ""answer"": ""uv run python -m app.ask を実行します。"",\n ""needs_review"": false\n}",○/○,
2,単発の質問に使う uv run のコマンドを教えてください,False,"{\n ""answer"": ""uv run python -m app.ask を使います。"",\n ""needs_review"": false\n}",○/○,
3,モデルを指定する環境変数は？,False,"{\n ""answer"": ""AIE_MODEL"",\n ""needs_review"": false\n}",○/○,
4,Gemini のモデル名を変えて実行するには何を使いますか,False,"answer: ""Gemini のモデル名を変えて実行するには uv run python -m app.ask を使います。使うモデルは AIE_MODEL という環境変数で切り替えます。""",×/×,json_error: JSONDecodeError
5,実行するモデルはどの環境変数で決まりますか,False,"{\n ""answer"": ""AIE_MODEL"",\n ""needs_review"": false\n}",○/○,
6,ツール一覧を出すオプションは？,False,"{""answer"": ""app.agent に --list-tools を付けると表示されます。""}",×/×,missing_key
7,app.agent で利用できるツールを表示するには？,False,"{\n ""answer"": ""app.agent で利用できるツールを表示するには、uv run python -m app.ask を使います。使うモデルは AIE_MODEL という環境変数で切り替えます。エージェントのツール一覧は app.agent に --list-tools を付けると表示されます。ツールは app.agent に --trace を付けて実行すると確認できます。各回の開始時点のコードは lecNN-start タグから作業ブランチを作って使います。"",",×/×,json_error: JSONDecodeError
8,エージェントのツール名をまとめて確認する方法は？,False,"{\n ""answer"": ""app.agent に --list-tools を付けるとツール一覧が表示されます。"",\n ""needs_review"": false\n}",○/○,
9,ツール呼び出しを表示するオプションは？,False,"{""answer"": ""app.agent に --trace を付けて実行すると確認できます。"", ""needs_review"": false}",○/○,
